[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/LTX-2-5_ComfyUI_Colab.ipynb) [View on GitHub](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/LTX-2-5_ComfyUI_Colab.ipynb)

**Open in Colab** ↑ for one-click run · **View on GitHub** ↑ for source



# 🎬 LTX-2.5 (ComfyUI runtime) — Text/Image-to-Video Generation

> **Runtime required:** GPU. **Recommended:** **L4 (24 GB)** for the FP8 distilled pipeline (~99 s per short clip after VRAM purge). **T4 (16 GB)** works with NVFP4 + aggressive offload but is materially slower per-step (no native FP8/FP4 tensor cores on Turing).
>
> When Colab asks you to pick a runtime, choose **L4** if available; fall back to T4 if not. The notebook auto-detects the GPU.

A Colab port of [Lightricks LTX-2.5](https://huggingface.co/Lightricks/LTX-2.5) — a 22B parameter distilled text/image-to-video model that produces **video with synchronized audio**, supports text-to-video, image-to-video, and video extension. Built on the [Lightricks/ComfyUI-LTXVideo](https://github.com/Lightricks/ComfyUI-LTXVideo) custom node pack.

**Workflow (L4 config) — matches Lightricks' example_workflows/2.5/:**

```
UNETLoader (transformer, ~20 GB int8-convrot distilled)
  → CFGGuider + KSamplerSelect + ManualSigmas
        EmptyLTXVLatentVideo (canvas dims, length frames)
        LTXVConditioning (positive prompt)
    → SamplerCustomAdvanced (8-step distilled, CFG=1, euler/normal)
        ↓ → LATENT
VAELoader (video VAE, 1.35 GB Conv)
  + VAELoader (audio VAE, 0.34 GB)
        ↓
LTXVAudioVAEDecode (audio stream)
VAEDecodeTiled     (video stream, 832x480 needs tiling)
        ↓
CreateVideo        (fuses A/V into one .mp4)
SaveVideo          (writes to COMFY_DIR/output/)
```

This is the workflow that Lightricks publishes as their canonical 2.5 distilled example (see `example_workflows/2.5/LTX-2.5_T2V_I2V_Single_Stage_Distilled.json`). It uses ComfyUI's standard UNETLoader + VAELoader (NOT the LowVRAMCheckpointLoader), and the LTXVGemmaCLIPModelLoader is replaced by `LTXVConditioning` from ComfyUI v0.31+ core — which builds the conditioning without loading the 14 GB Gemma 3 text encoder locally.

**Why not guillaume127/LTX-2.5-FP8 (the @TIMES99 / the README's path)?** guillaume's FP8 distilation reads as a diffusion-only state_dict (no `model.diffusion_model.` prefix), so it loads via `UNETLoader` from `models/diffusion_models/`. That part works. **The bottleneck is Gemma + UNET together:** 14.32 GB + 21.87 GB = **36.19 GB**, more than 24 GB. guillaume's README recommends a VRAM Cleanup node or `--highvram`; ComfyUI v0.32 has neither (`--highvram` is launch-only and assumes 32+ GB total). For 24 GB Colab, the int8-convrot distilled checkpoint + LTXVConditioning (no Gemma) is the path that actually fits.

**For systems with 32+ GB VRAM:** the user can switch `TRANSFORMER = guillaume127/LTX-2.5-FP8` in STEP 2 to get the FP8 weights; the workflow nodes are unchanged. They will then see ~99 s/clip as guillaume measured on RTX 4090.

**For LLM-enhanced prompts:** swap `LTXVConditioning` (node 10) for `LTXVGemmaCLIPModelLoader + LTXVGemmaEnhancePrompt + CLIPTextEncode`. This re-adds the 14 GB Gemma 3 weight — the user can also enable `LTXVGemmaCLIPModelLoader` by adding the gemma4-12b file from `Lightricks/LTX-2.5/text_encoders/` (the form would need a "Use local Gemma" toggle).

## Companion components (in STEP 2)

Every LTX-2.5 workflow loads three supporting files in addition to the transformer:

**Text encoder** (`TEXT_ENCODER`): Gemma 4 12B with LTX's `text_embedding_projection` + `audio_projector` layers baked in. Two variants on `Lightricks/LTX-2.5/text_encoders/`:

```
gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors  15.4 GB  - default; fits 24 GB alongside the transformer
gemma4-12b-with-proj-ltx-2.5-bf16.safetensors               26.3 GB  - full precision; for 32+ GB hardware only
```

**This file is only loaded when the workflow uses `LTXVGemmaCLIPModelLoader` (the optional LLM prompt-expansion path).** The default workflow uses `LTXVConditioning` which does NOT load this file — the bundled ComfyUI CLIP + LTXV text encoder intrinsic handle conditioning in ~2 GB. So for the standard 24 GB path, the text-encoder download size does NOT add to runtime VRAM. It only matters if you swap in the Gemma LLM expansion path.

**Video VAE** (`VIDEO_VAE`): decodes the latent frames back to pixels. Two variants:

```
ltx-2.5-video-vae-conv-bf16.safetensors  1.45 GB  - Conv variant; faster decode; default
ltx-2.5-video-vae-bf16.safetensors       1.47 GB  - DiffVAE variant; marginally higher fidelity
```

The two VAEs are nearly indistinguishable at 832x480 and below. The DiffVAE becomes worth switching to only at 1504x832+ where its reconstruction sharpness matters. **The audio VAE** (`ltx-2.5-audio-vae-bf16.safetensors`, 0.34 GB) is auto-loaded and not configurable — it's small enough to always be present.

**Recommended for L4 (24 GB):** TEXT_ENCODER = int8-convrot, VIDEO_VAE = Conv.
**Recommended for T4 (16 GB):** Same encoder, Conv VAE. Avoid the DiffVAE at T4 — it costs 0.02 GB but uses more decode CUDA time which can push you past session timeout.

## GGUF quants (optional, lower VRAM)

Set `TRANSFORMER` in STEP 2 to one of the GGUF choices and pick a `QUANT` (Q2 through Q8):

```
realrebelai/LTX-2.5_GGUFs              Q2_K=8.83 GB, Q3_K_M=11.5, Q4_K_M=15.1, Q6_K=18.7, Q8_0=23.6
Abiray/LTX-2.5-Distilled-GGUF          Q3_K_S=12.6 (only here), Q4_K_M=15.7, Q6_K=18.6, Q8_0=23.6
ChrisColeTech/LTX-2.5-turbo-GGUF        (no auto-download; uses an existing GGUF in models/unet/ via CCTech custom loader)
```

GGUF files live in `models/unet/` (not `models/diffusion_models/`) and load via **Unet Loader (GGUF)** from the `city96/ComfyUI-GGUF` custom node pack (cloned in STEP 1). Same workflow nodes downstream (`EmptyLTXVLatentVideo`, `LTXVConditioning`, `CFGGuider`, `SamplerCustomAdvanced`, `LTXVAudioVAEDecode`, `VAEDecodeTiled`, `CreateVideo`, `SaveVideo`).

**Pick GGUF when:**
- You have a 16 GB T4 and want LTX-2.5 to fit (Q2_K ~9 GB dist).
- You have a 24 GB L4 but want to leave headroom for longer contexts/longer clips (Q4_K_M ~15 GB vs 20 GB int8-convrot).
- You want the smallest Drive footprint (Q2_K uses ~9 GB, less than half the int8-convrot dist).

**Stay on safetensors when:**
- You want maximum visual fidelity (Q8_0 ~24 GB ≈ int8-convrot; not enough gain to switch).
- You want the built-in ComfyUI `UNETLoader` (no extra custom node).

**Tradeoffs:** GGUF quantization can subtly desync the audio stream at aggressive quants (`<Q4_K_M`) — the model card from realrebelai documents this and preserves `to_gate_logits` at higher precision to mitigate it. Q4_K_M and above are visually and audibly indistinguishable from bf16 for most prompts; Q3_K_M starts to lose fine detail. Q2_K is the smallest but the quality drop is visible.

## ⚠️ License

LTX-Video Open Weights License v0.1 — permits non-commercial research and personal use; commercial use requires a separate license. See [Lightricks/LTX-Video](https://huggingface.co/Lightricks/LTX-Video/blob/main/LTX-Video-Open-Weights-License-0.X.txt).



In [ ]:
#@title STEP 1 — Install ComfyUI + ComfyUI-LTXVideo + ComfyUI-GGUF (Drive-persistent)

"""
• Mounts Google Drive for the weights cache + ComfyUI installation
• Clones ComfyUI v0.30.1+ to /content/drive/MyDrive/AEI_ComfyUI/ (shared
  with MiniMax-H3 notebook — same install root, model-specific weights
  live in /content/drive/MyDrive/AEI_3D_Cache/LTX-Video-2.5/weights/)
• Installs torch 2.11.0+cu130 (L4 / T4 / A100 compatible)
• Installs ComfyUI's requirements.txt (transformers, tokenizers, safetensors, av, ...)
• Installs ComfyUI-LTXVideo's requirements.txt (diffusers, einops, ninja, ...)
• Clones Lightricks/ComfyUI-LTXVideo custom node pack into custom_nodes/
• Sets PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to reduce fragmentation
"""

import os, sys, subprocess, time, pathlib
from pathlib import Path

print('='*72)
print('LTX-2.5 / ComfyUI — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  Python : {sys.version.split()[0]}')
    print(f'  torch  : {torch.__version__}  CUDA: {torch.version.cuda}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU    : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected')
except ImportError:
    print('  torch not yet installed')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path('/content/_ltx_cache')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

COMFY_DIR = DRIVE_ROOT / 'AEI_ComfyUI'
HF_CACHE  = DRIVE_ROOT / 'AEI_3D_Cache' / 'LTX-Video-2.5'
OUT_DIR   = DRIVE_ROOT / 'AEI_3D_Out' / 'LTX-Video-2.5'
COMFY_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ComfyUI/custom_nodes/ might not exist yet - create it explicitly
# before any git clone so Drive FUSE doesn't fail the clone at the
# intermediate-dir step (exit 255 with empty stderr).
(COMFY_DIR / 'custom_nodes').mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME']               = str(HF_CACHE)
os.environ['HUGGINGFACE_HUB_CACHE']  = str(HF_CACHE)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.8')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

print(f'  Drive cache  : {HF_CACHE}')
print(f'  ComfyUI dir  : {COMFY_DIR}')
print(f'  Output dir   : {OUT_DIR}')

if not COMFY_DIR.joinpath('main.py').exists():
    print(f'  Cloning ComfyUI to {COMFY_DIR} ...')
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY_DIR)], check=True)
else:
    print(f'  Reusing existing {COMFY_DIR}')

print('  Installing pytorch cu130 ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'torch', 'torchvision', 'torchaudio',
                '--index-url', 'https://download.pytorch.org/whl/cu130'], check=False)

# tqdm is used by huggingface_hub's snapshot_download progress bar; without
# it, downloads quietly print "Downloading: 100%" with no rate/ETA. Install
# it explicitly so the 20 GB transformer pull doesn't go dark for 4-8 min.
print('  Installing tqdm + hf_transfer for progress bars ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'tqdm', 'hf_transfer'], check=False)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

print('  Installing ComfyUI requirements ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '-r', str(COMFY_DIR / 'requirements.txt')], check=False)

# ComfyUI-LTXVideo's requirements pulls diffusers + einops + ninja +
# transformers>=4.50 (Gemma 3 needs newer transformers). Install
# unconditionally since pip is idempotent.
print('  Installing ComfyUI-LTXVideo requirements ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'diffusers', 'einops', 'kornia',
                'transformers[timm]>=4.50.0',
                'huggingface_hub>=0.25.2'], check=False)

# Clone ComfyUI-LTXVideo (Lightricks' official LTX node pack) into
# custom_nodes/. Same pattern as the MiniMax-H3 notebook's kjnodes clone
# — only clone on first run, reuse after.
_LTX_DIR = COMFY_DIR / 'custom_nodes' / 'ComfyUI-LTXVideo'
# If _LTX_DIR exists but has no .git (e.g. previous clone
# crashed mid-write), git refuses to clone into a non-empty
# dir. Treat that as "needs re-clone" by checking for __init__.py
# - that's ComfyUI's signature for 'this is a custom node'.
if not _LTX_DIR.exists() or not (_LTX_DIR / "__init__.py").exists():
    if _LTX_DIR.exists():
        print(f'  Removing partial clone at {_LTX_DIR} ...')
        import shutil as _sh
        _sh.rmtree(_LTX_DIR)
    print(f'  Cloning ComfyUI-LTXVideo to {_LTX_DIR} ...')
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/Lightricks/ComfyUI-LTXVideo.git',
                    str(_LTX_DIR)], check=True)
    _req = _LTX_DIR / 'requirements.txt'
    if _req.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        '-r', str(_req)], check=False)
else:
    print(f'  Reusing existing {_LTX_DIR}')

# Clone city96/ComfyUI-GGUF (the GGUF loader custom node). This
# adds 'Unet Loader (GGUF)' + 'CLIPLoaderGGUF' + 'VAELoaderGGUF'
# nodes. Only needed if you pick a GGUF transformer in STEP 2,
# but small (<5 MB) so we always install it — that's simpler
# than a mode-gated install. city96 is the canonical ComfyUI
# GGUF integration; the README docs and workflow JSON files
# from realrebelai/LTX-2.5_GGUFs + Abiray/LTX-2.5-Distilled-GGUF
# assume it.
_GGUF_DIR = COMFY_DIR / 'custom_nodes' / 'ComfyUI-GGUF'
if not _GGUF_DIR.exists() or not (_GGUF_DIR / "__init__.py").exists():
    if _GGUF_DIR.exists():
        print(f'  Removing partial clone at {_GGUF_DIR} ...')
        import shutil as _sh
        _sh.rmtree(_GGUF_DIR)
    print(f'  Cloning ComfyUI-GGUF to {_GGUF_DIR} ...')
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/city96/ComfyUI-GGUF.git',
                    str(_GGUF_DIR)], check=True)
    _greq = _GGUF_DIR / 'requirements.txt'
    if _greq.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        '-r', str(_greq)], check=False)
else:
    print(f'  Reusing existing {_GGUF_DIR}')

# ChrisColeTech/LTX-2.5-turbo-GGUF requires 'CCTech Unet Loader'
# (a custom node not on city96's standard loader list). Install via
# ComfyUI Manager: switch channel to 'remote' and install
# 'comfyui-gguf-loader' — that's the pack ChrisColeTech's README
# references. Only needed if you set TRANSFORMER to
# 'ChrisColeTech/LTX-2.5-turbo-GGUF' in STEP 2.

import torch
print(f'  torch        : {torch.__version__}  (CUDA {torch.version.cuda})')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'  GPU          : {p.name}  ({p.total_memory / 1024**3:.1f} GB)')
    print(f'  Compute      : {p.major}.{p.minor}')
else:
    raise SystemExit('No GPU detected - ComfyUI needs CUDA.')



In [ ]:
#@title STEP 2 — Download LTX-2.5 weights to Drive cache

"""
• Downloads based on the TRANSFORMER dropdown choice:
    - Lightricks/LTX-2.5 (int8-convrot distilled, 20 GB) — default; works on 24 GB L4
    - Lightricks/LTX-2.5 (nvfp4 distilled, 17 GB) — slightly tighter VRAM
    - guillaume127/LTX-2.5-FP8 (FP8, 21.87 GB) — keeps stock UNETLoader
    - realrebelai/LTX-2.5_GGUFs (Q2..Q8 GGUF distilled, 8.83..23.6 GB)
    - Abiray/LTX-2.5-Distilled-GGUF (Q2..Q8 GGUF distilled, adds Q3_K_S)
    - ChrisColeTech/LTX-2.5-turbo-GGUF — no auto-fetch; user supplies GGUF
      in models/unet/ (drop via ComfyUI Manager or a separate download)
• Common supporting files for all choices:
    - ltx-2.5-video-vae-conv-bf16 (video VAE, 1.35 GB)
    - ltx-2.5-audio-vae-bf16 (audio VAE, 0.34 GB)
    - ltx-2.5-latent-spatial-upscaler-x2 (0.93 GB; still required for the
      2.5 distilled pipeline per Lightricks release notes)
    - (optional) gemma4-12b-with-proj-ltx-2.5 text encoder (14 GB) — only
      needed if you swap LTXVConditioning for GemmaCLIPModelLoader +
      GemmaEnhancePrompt for LLM-enhanced prompts.
• Resolves expected sizes from HF manifest; HEAD-fallback for files
  where the manifest leaves size=None (older huggingface_hub versions).
• Per-repo fetches, then symlinks into ComfyUI/models/{diffusion_models,unet,vae}.
  Safetensors transformers (Lightricks/int8, nvfp4, guillaume FP8) live in
  models/diffusion_models/ and load via UNETLoader. GGUF transformers (any
  *_GGUF choice) live in models/unet/ and load via Unet Loader (GGUF) from
  city96/ComfyUI-GGUF (cloned in STEP 1).
"""
import os, sys, time, subprocess, urllib.request
from pathlib import Path

HF_CACHE = Path('/content/drive/MyDrive/AEI_3D_Cache/LTX-Video-2.5')
HF_CACHE.mkdir(parents=True, exist_ok=True)

# Weights selection — mode toggle chooses the transformer file. The
# text encoder + VAEs + upscaler are model-level (one per model, not
# per-mode). Same pattern as the MiniMax-H3 notebook's STEPS-default
# mode-routing (per-mode constants resolved to literals at build time,
# Colab's #@param parser requires Python expressions).
# Default 'Lightricks int8-convrot' — the only LTX checkpoint that fits
# cleanly into the standard UNETLoader + VAELoader workflow AND works on
# 24 GB Colab cards (the FP8 and nvfp4 variants require either --highvram
# or a non-existent VRAM-cleanup node to avoid PCIe thrashing on 24 GB).
# User can override with 'guillaume127/LTX-2.5-FP8' once they verify their
# hardware can fit text encoder + transformer + VAE in 32+ GB.
# Select the diffusion transformer. Default is the Lightricks
# distilled int8-convrot (only one that fits 24 GB Colab as
# UNETLoader with stock Gemma + Conv VAE). The GGUF options
# use Llama.cpp-style quantization (Q2..Q8) loaded via
# 'Unet Loader (GGUF)' from city96/ComfyUI-GGUF. GGUF quants
# run at lower VRAM than the equivalent safetensors variant
# and work on cards where the int8-convrot path doesn't fit.
# "CCTech Unet Loader" (ChrisColeTech's turbo variant) needs
# comfyui-gguf-loader installed via ComfyUI Manager remote.
# Select the diffusion transformer. Defaults to Lightricks
# distilled int8-convrot (only one that fits 24 GB Colab as
# UNETLoader with stock Gemma + Conv VAE). GGUF options use
# Llama.cpp-style quantization (Q2..Q8) loaded via
# 'Unet Loader (GGUF)' from city96/ComfyUI-GGUF. GGUF quants
# run at lower VRAM than the safetensors equivalent and work
# on cards where the int8-convrot path doesn't fit. .Note: Python
# list literals cannot span lines, so the dropdown has to be a single
# line - Colab's #@param parser is strict.
# Python list literals cannot span lines, so the dropdown has
# to be a single line — Colab's #@param parser is strict.
TRANSFORMER = 'Lightricks/LTX-2.5 (distilled int8-convrot)'  #@param ["Lightricks/LTX-2.5 (distilled int8-convrot)", "Lightricks/LTX-2.5 (distilled nvfp4)", "guillaume127/LTX-2.5-FP8", "realrebelai/LTX-2.5_GGUFs (distilled GGUF)", "Abiray/LTX-2.5-Distilled-GGUF (distilled GGUF)", "ChrisColeTech/LTX-2.5-turbo-GGUF (uses Lightricks distilled GGUF)"] {"allow-input": true}
# Quant level for the GGUF options. Ignored for the int8-convrot,
# nvfp4, and FP8 safetensors transformers. Q4_K_M is the most
# common recommendation (best size/quality tradeoff); Q2_K is
# the smallest (~9 GB, fits 16 GB T4 if you disable most VAEs).
QUANT = "Q4_K_M"  #@param ["Q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "Q4_K_S", "Q3_K_M", "Q2_K"] {"allow-input": true}
# Text encoder (Gemma 4 12B with LTX custom projection layers).
#   int8-convrot = 15.4 GB  - quantized for low VRAM; default
#   bf16         = 26.3 GB  - full precision; for 32+ GB hardware
# Most users should leave this at 'int8-convrot'. The bf16 variant
# only matters if you have 32+ GB and want marginally better prompt
# adherence (rarely worth the 11 GB extra). NOTE: this text encoder
# is only loaded if the workflow uses 'LTXVGemmaCLIPModelLoader' (the
# LLM-prompt-expansion path). The default workflow uses
# LTXVConditioning which does NOT load this file, so for the
# standard 24 GB path this dropdown has no runtime cost.
TEXT_ENCODER = "gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot"  #@param ["gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot (int8 + convrot, 15.4 GB, default for 24 GB)", "gemma4-12b-with-proj-ltx-2.5-bf16 (bf16, 26.3 GB, for 32+ GB)"] {"allow-input": true}
TEXT_ENCODER = TEXT_ENCODER.split(" (")[0]
# Video VAE (decodes the latent -> pixels for video frames).
#   Conv = convolutional variant, 1.45 GB - faster decode, default
#   bf16 = DiffVAE,              1.47 GB - marginally higher fidelity
# The two variants are nearly indistinguishable on 832x480 output;
# the Conv VAE is ~10-15% faster at decode. The DiffVAE becomes
# worth switching to only at the highest resolutions (1504x832 and up).
VIDEO_VAE = "ltx-2.5-video-vae-conv-bf16"  #@param ["ltx-2.5-video-vae-conv-bf16 (Conv VAE, 1.45 GB, recommended)", "ltx-2.5-video-vae-bf16 (DiffVAE, 1.47 GB, marginally higher fidelity)"] {"allow-input": true}
VIDEO_VAE = VIDEO_VAE.split(" (")[0]
USE_UPSCALER = True  #@param {type:"boolean"}
print('='*72)
print('LTX-2.5 / ComfyUI — Download weights')
print('='*72)
print(f'  Transformer   : {TRANSFORMER}')
print(f'  Text encoder  : {TEXT_ENCODER}')
print(f'  Video VAE     : {VIDEO_VAE}')
print(f'  Spatial upscaler : {USE_UPSCALER}')
print()

import os
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_CACHE)

from huggingface_hub import HfApi, snapshot_download

# Map the user-friendly TRANSFORMER dropdown to a concrete filename +
# the upstream repo. The third option (Lightricks nvfp4) is on the
# same Lightricks/LTX-2.5 repo — just a different filename.
_TRANSFORMER_PATHS = {
    'Lightricks/LTX-2.5 (distilled int8-convrot)':
        ('Lightricks/LTX-2.5',
         'diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors'),
    'Lightricks/LTX-2.5 (distilled nvfp4)':
        ('Lightricks/LTX-2.5',
         'diffusion_models/ltx-2.5-22b-distilled-transformer-nvfp4.safetensors'),
    'guillaume127/LTX-2.5-FP8':
        ('guillaume127/LTX-2.5-FP8',
         'ltx-2.5-22b-distilled-transformer-fp8_e4m3fn.safetensors'),
}
# GGUF transformer metadata: (repo_id, filename_in_repo, scope_path).
# The filename varies by QUANT; scope_path is 'unet' for GGUF
# (city96 loader convention) or 'diffusion_models' for safetensors.
_GGUF_FILES = {
    'realrebelai/LTX-2.5_GGUFs': {
        'Q8_0':    'LTX-2.5-Distilled-Q8_0.gguf',
        'Q6_K':    'LTX-2.5-Distilled-Q6_K.gguf',
        'Q5_K_M':  'LTX-2.5-Distilled-Q5_K_M.gguf',
        'Q4_K_M':  'LTX-2.5-Distilled-Q4_K_M.gguf',
        'Q4_K_S':  'LTX-2.5-Distilled-Q4_K_S.gguf',
        'Q3_K_M':  'LTX-2.5-Distilled-Q3_K_M.gguf',
        'Q2_K':    'LTX-2.5-Distilled-Q2_K.gguf',
    },
    'Abiray/LTX-2.5-Distilled-GGUF': {
        'Q8_0':    'LTX-2.5-Distilled-Q8_0.gguf',
        'Q6_K':    'LTX-2.5-Distilled-Q6_K.gguf',
        'Q5_K_M':  'LTX-2.5-Distilled-Q5_K_M.gguf',
        'Q4_K_M':  'LTX-2.5-Distilled-Q4_K_M.gguf',
        'Q4_K_S':  'LTX-2.5-Distilled-Q4_K_S.gguf',
        'Q3_K_M':  'LTX-2.5-Distilled-Q3_K_M.gguf',
        # Abiray ships Q3_K_S too — bonus over realrebelai.
        'Q3_K_S':  'LTX-2.5-Distilled-Q3_K_S.gguf',
    },
    'ChrisColeTech/LTX-2.5-turbo-GGUF': {
        # Repo currently holds VAE + upscaler split files but no
        # transformer as of 2026-08-12. We don't auto-download from
        # ChrisColeTech (nothing transformer-shaped there). If the
        # user picks this TRANSFORMER, they need to have already
        # dropped a GGUF (e.g. Q4_K_M from realrebelai) into
        # models/unet/ via a prior run. The 'filename' here is the
        # one city96-GGUF-style loader expects.
        'Q8_0':    'LTX-2.5-Distilled-Q8_0.gguf',
        'Q6_K':    'LTX-2.5-Distilled-Q6_K.gguf',
        'Q5_K_M':  'LTX-2.5-Distilled-Q5_K_M.gguf',
        'Q4_K_M':  'LTX-2.5-Distilled-Q4_K_M.gguf',
        'Q4_K_S':  'LTX-2.5-Distilled-Q4_K_S.gguf',
        'Q3_K_M':  'LTX-2.5-Distilled-Q3_K_M.gguf',
        'Q3_K_S':  'LTX-2.5-Distilled-Q3_K_S.gguf',
        'Q2_K':    'LTX-2.5-Distilled-Q2_K.gguf',
    },
}
# Which ComfyUI loader class to use, and whether the file is
# GGUF (load via 'Unet Loader (GGUF)' from city96, scope=unet) or
# safetensors (load via stock UNETLoader, scope=diffusion_models).
_LOADER_BY_TRANSFORMER = {
    'realrebelai/LTX-2.5_GGUFs (distilled GGUF)':
        ('realrebelai/LTX-2.5_GGUFs', 'unet', 'UnetLoaderGGUF'),
    'Abiray/LTX-2.5-Distilled-GGUF (distilled GGUF)':
        ('Abiray/LTX-2.5-Distilled-GGUF', 'unet', 'UnetLoaderGGUF'),
    'ChrisColeTech/LTX-2.5-turbo-GGUF (uses Lightricks distilled GGUF)':
        # Same repo ID for fetch as realrebelai (CCTech has no
        # transformer file currently) but loader kind is CCTech's
        # custom node. Falls back to 'UnetLoaderGGUF' if CCTech's
        # custom node isn't installed.
        ('realrebelai/LTX-2.5_GGUFs', 'unet', 'CCTechUnetLoader'),
}
_loader = _LOADER_BY_TRANSFORMER.get(TRANSFORMER)
if _loader:
    _gg_repo, _scope, _loader_class = _loader
    if TRANSFORMER.startswith('ChrisColeTech'):
        # CCTech repo has no transformer; use realrebelai's file
        # but reference the CCTech loader (node class). The user
        # is responsible for installing the loader via ComfyUI Manager.
        _t_repo = _gg_repo
        _t_filename = _GGUF_FILES['ChrisColeTech/LTX-2.5-turbo-GGUF'][QUANT]
    else:
        _t_repo = _gg_repo
        # _GGUF_FILES keys are repo-id bases (not the dropdown label)
        _base_repo_key = TRANSFORMER.split(' (')[0]
        _t_filename = _GGUF_FILES[_base_repo_key][QUANT]
else:
    _t_repo, _t_filename = _TRANSFORMER_PATHS[TRANSFORMER]
    _scope = 'diffusion_models'
    _loader_class = 'UNETLoader'
print(f'  Resolving sizes from HF manifests ...')
_expected = {}
# Resolve HF manifest sizes for every repo we might pull from.
_repos_to_query = {
    _t_repo,
    'Lightricks/LTX-2.5',
    'guillaume127/LTX-2.5-FP8',
    'realrebelai/LTX-2.5_GGUFs',
    'Abiray/LTX-2.5-Distilled-GGUF',
    'ChrisColeTech/LTX-2.5-turbo-GGUF',
}
for _repo in _repos_to_query:
    try:
        info = HfApi().repo_info(_repo, files_metadata=True)
        for sib in info.siblings:
            if sib.size is not None:
                _expected[(_repo, sib.rfilename)] = sib.size
    except Exception as e:
        print(f'  Could not resolve {_repo} sizes: {e}')

def _size(repo, fn, fallback_path=None):
    sz = _expected.get((repo, fn))
    if sz is not None:
        return sz
    print(f'  size missing in manifest; HEAD-resolving {repo}/{fn} ...')
    url = f'https://huggingface.co/{repo}/resolve/main/{fn}'
    if fallback_path:
        url = f'https://huggingface.co/{repo}/resolve/main/{fallback_path}/{fn}'
    req = urllib.request.Request(url, method='HEAD')
    with urllib.request.urlopen(req, timeout=30) as r:
        return int(r.headers['content-length'])

_files_to_fetch = [
    ('Lightricks/LTX-2.5', f'text_encoders/{TEXT_ENCODER}.safetensors', 'text_encoders'),
    (_t_repo, _t_filename, _scope),
    ('Lightricks/LTX-2.5', f'vae/{VIDEO_VAE}.safetensors', 'vae'),
    ('Lightricks/LTX-2.5', 'vae/ltx-2.5-audio-vae-bf16.safetensors', 'vae'),
]
if USE_UPSCALER:
    _files_to_fetch.append(
        ('Lightricks/LTX-2.5', 'latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors', 'latent_upscale_models')
    )

# Build per-repo fetch plans. The main Lightricks/LTX-2.5 repo
# holds the text encoders + VAEs + most transformer variants;
# GGUF transformers live in realrebelai/Abiray/ChrisColeTech.
# guillaume127 FP8 is its own repo. We bucket downloads by repo
# and call snapshot_download once per repo (which is the supported
# way - Hugging Face download paths are per-repo).
_PATTERNS_BY_REPO = {}
for _repo, _fn, _subdir in _files_to_fetch:
    _PATTERNS_BY_REPO.setdefault(_repo, []).append(
        f'{_subdir}/{_fn}' if _subdir == 'diffusion_models' else _fn
    )
print(f'  Fetching from {len(_PATTERNS_BY_REPO)} repo(s) ...')
t0 = time.time()
for _repo, _patterns in _PATTERNS_BY_REPO.items():
    print(f'    {_repo}: {len(_patterns)} file(s) ...')
    try:
        snapshot_download(
            repo_id=_repo,
            local_dir=str(HF_CACHE),
            allow_patterns=_patterns,
        )
    except Exception as e:
        print(f'    WARN: download from {_repo} failed: {e}')
        print(f'          Re-run STEP 2 to retry (HF cache is resumable).')
print(f'  Weights cached at {HF_CACHE} in {time.time() - t0:.0f}s')
print()
# Map HF_CACHE subdirs into ComfyUI/models/<subdir>/<file>.
# snapshot_download left the files under HF_CACHE/<subdir>;
# ComfyUI scans <install_root>/models/<subdir>/. We symlink so
# Drive quota isn't doubled (ComfyUI follows symlinks when
# loading safetensors - it just opens the file path).
DIFF = COMFY_DIR
DIFF = DIFF / 'models' / 'diffusion_models'
TXT  = COMFY_DIR / 'models' / 'text_encoders'
VAE  = COMFY_DIR / 'models' / 'vae'
UPS  = COMFY_DIR / 'models' / 'latent_upscale_models'
for d in (DIFF, TXT, VAE, UPS):
    d.mkdir(parents=True, exist_ok=True)

# Symlink helper: symlinks save Drive space and ComfyUI follows symlinks
# when loading safetensors (it just opens the file, doesn't care whether
# it's a real file or a symlink). Re-runs after a download are idempotent.
def _ln(src: Path, dst: Path):
    if dst.exists() or dst.is_symlink():
        return
    os.symlink(str(src), str(dst))
_links = [
    (HF_CACHE / 'text_encoders' / f'{TEXT_ENCODER}.safetensors',
     TXT / f'{TEXT_ENCODER}.safetensors'),
    # Transformer: scope='diffusion_models' (safetensors) goes to DIFF,
    # scope='unet' (GGUF) goes to UNET. Snapshot_download put the
    # safetensors under HF_CACHE/diffusion_models/ but GGUF under
    # HF_CACHE/ (repo root, no subdir) since realrebelai + Abiray + 
    # ChrisColeTech all publish at root. Either way, Path(_t_filename).name
    # is the basename we need to link into ComfyUI's expected subdir.
    (HF_CACHE / 'diffusion_models' / Path(_t_filename).name if _scope == 'diffusion_models'
     else HF_CACHE / Path(_t_filename).name,
     DIFF / Path(_t_filename).name if _scope == 'diffusion_models'
     else UNET / Path(_t_filename).name),
    (HF_CACHE / 'vae' / f'{VIDEO_VAE}.safetensors', VAE / f'{VIDEO_VAE}.safetensors'),
    (HF_CACHE / 'vae' / 'ltx-2.5-audio-vae-bf16.safetensors',
     VAE / 'ltx-2.5-audio-vae-bf16.safetensors'),
]
if USE_UPSCALER:
    _links.append(
        (HF_CACHE / 'latent_upscale_models' / 'ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors',
         UPS / 'ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors')
    )
# Expose the TRANSFORMER + QUANT choices to STEPS 6/7/8.5 via builtins.
# These cells read them back to choose UNETLoader vs UnetLoaderGGUF
# vs CCTechUnetLoader and the matching model filename.
import builtins as _b
_b._TRANSFORMER_VAR = TRANSFORMER
_b._QUANT_VAR = QUANT
_b._SCOPE_VAR = _scope
print(f'  Exposed builtins._TRANSFORMER_VAR = {TRANSFORMER}')
print(f'  Exposed builtins._QUANT_VAR = {QUANT}')
print(f'  Exposed builtins._SCOPE_VAR = {_scope}')
for src, dst in _links:
    _ln(src, dst)
print(f'  Symlinked {len(_links)} files into ComfyUI/models/.')

# Verify sizes against HF manifest (skip if missing — manifests vary).
def _expected_bytes(path: Path) -> int:
    rl = path.relative_to(HF_CACHE).as_posix()
    if rl.startswith('text_encoders/'):
        return _size('Lightricks/LTX-2.5', TEXT_ENCODER + '.safetensors')
    if 'ltx-2.5-22b-distilled-transformer-fp8_e4m3fn.safetensors' in rl:
        return _size('guillaume127/LTX-2.5-FP8', 'ltx-2.5-22b-distilled-transformer-fp8_e4m3fn.safetensors')
    if 'transformer-comfy-int8-convrot.safetensors' in rl:
        return _size('Lightricks/LTX-2.5',
                     'diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors')
    if 'transformer-nvfp4.safetensors' in rl:
        return _size('Lightricks/LTX-2.5',
                     'diffusion_models/ltx-2.5-22b-distilled-transformer-nvfp4.safetensors')
    if 'transformer-fp8_e4m3fn.safetensors' in rl:
        return _size('Lightricks/LTX-2.5',
                     'diffusion_models/ltx-2.5-22b-distilled-transformer-fp8_e4m3fn.safetensors')
    if rl.endswith('video-vae-conv-bf16.safetensors') or rl.endswith('video-vae-bf16.safetensors'):
        return _size('Lightricks/LTX-2.5', f'vae/{VIDEO_VAE}.safetensors')
    if rl.endswith('audio-vae-bf16.safetensors'):
        return _size('Lightricks/LTX-2.5', 'vae/ltx-2.5-audio-vae-bf16.safetensors')
    # GGUF transformer symlinks: GT replaced 'texturettx' in HF cache
    # (root, no subdir) with HF_CACHE / <gguf-filename>. Both
    # realrebelai and Abiray publish at repo root.
    if rl.endswith('.gguf'):
        # Filename like 'LTX-2.5-Distilled-Q4_K_M.gguf' - the QUANT
        # is the last token before '.gguf'. We resolve the expected
        # size by HF manifest lookup against the appropriate repo.
        _gguf_repo = 'realrebelai/LTX-2.5_GGUFs'
        if _t_repo in ('Abiray/LTX-2.5-Distilled-GGUF',):
            _gguf_repo = 'Abiray/LTX-2.5-Distilled-GGUF'
        elif _t_repo == 'ChrisColeTech/LTX-2.5-turbo-GGUF':
            # CCTech currently holds no transformer; their files are
            # VAE + upscaler split files we wouldn't reach here.
            _gguf_repo = 'realrebelai/LTX-2.5_GGUFs'
        _fn = Path(rl).name
        return _size(_gguf_repo, _fn)
    if 'spatial-upscaler' in rl:
        return _size('Lightricks/LTX-2.5',
                     'latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors')
    return -1

WARNINGS = []
for _src, _dst in _links:
    if not _dst.exists():
        WARNINGS.append(f'  ! symlink target missing: {_dst}')
        continue
    _sz_dst = _dst.resolve().stat().st_size
    _sz_exp = _expected_bytes(_src)
    if _sz_exp > 0 and _sz_dst != _sz_exp:
        WARNINGS.append(
            f'  ! Size mismatch for {_src.name}: have {_sz_dst/1024**3:.2f} GB, '
            f'expected {_sz_exp/1024**3:.2f} GB. Re-run STEP 2 to re-download.'
        )
if WARNINGS:
    print('Filesystem warnings:')
    for w in WARNINGS:
        print(w)
else:
    print('  All weights verified.')

# Print a summary table of what's now on disk.
print()
print('Final layout:')
# Expose the TRANSFORMER + QUANT choices to STEPS 6/7/8.5 via builtins.
# These cells read them back to choose UNETLoader vs UnetLoaderGGUF
# vs CCTechUnetLoader and the matching model filename.
import builtins as _b
_b._TRANSFORMER_VAR = TRANSFORMER
_b._QUANT_VAR = QUANT
_b._SCOPE_VAR = _scope
print(f'  Exposed builtins._TRANSFORMER_VAR = {TRANSFORMER}')
print(f'  Exposed builtins._QUANT_VAR = {QUANT}')
print(f'  Exposed builtins._SCOPE_VAR = {_scope}')
for src, dst in _links:
    print(f'  {dst} -> {src}')



In [ ]:
#@title STEP 3 — Launch ComfyUI subprocess (--disable-pinned-memory)

"""
Standard AEI-ComfyUI launch — same pattern as MiniMax-H3 notebook.
The --disable-pinned-memory + --fp16-intermediates + --disable-api-nodes
flags are the model-agnostic recipe that lets the 22B LTX-2.5 pipeline
fit in 24 GB Colab tiers without host-RAM OOM.
"""
import os, sys, time, subprocess, urllib.request, urllib.error

COMFY_DIR = Path('/content/drive/MyDrive/AEI_ComfyUI')
COMFY_HOST = '127.0.0.1'
COMFY_PORT = 8188
COMFY_URL = f'http://{COMFY_HOST}:{COMFY_PORT}'

LAUNCH_CMD = [
    sys.executable, 'main.py',
    '--listen', COMFY_HOST,
    '--port', str(COMFY_PORT),
    '--disable-pinned-memory',
    '--fp16-intermediates',
    '--disable-api-nodes',
    '--output-directory', str(COMFY_DIR / 'output'),
    '--input-directory', str(COMFY_DIR / 'input'),
]
# Auto-detect compute capability; Turing (sm_75) gets the VRAM-reserving flags.
import torch as _torch_check
_GPU_CC_AUTO = None
if _torch_check.cuda.is_available():
    _gp_auto = _torch_check.cuda.get_device_properties(0)
    _GPU_CC_AUTO = float(f'{_gp_auto.major}.{_gp_auto.minor}')
    if _GPU_CC_AUTO < 8.0:
        LAUNCH_CMD.extend(['--reserve-vram', '0.9'])
        LAUNCH_CMD.append('--fast-disk')
        print(f'  GPU compute capability {_GPU_CC_AUTO} (Turing) detected — '
              'auto-enabled --reserve-vram 0.9 --fast-disk.')
    else:
        print(f'  GPU compute capability {_GPU_CC_AUTO} (Ampere/Ada/Hopper) — '
              'no extra launch flags needed.')
env = os.environ.copy()

# Kill leftover ComfyUI from prior runs (pkill + fuser pattern).
subprocess.run(['pkill', '-9', '-f', f'{COMFY_DIR.name}.*main.py'], check=False)
time.sleep(2)
for _cmd in (['fuser', '-k', f'{COMFY_PORT}/tcp'], ['lsof', '-ti', f':{COMFY_PORT}']):
    try:
        r = subprocess.run(_cmd, capture_output=True, timeout=5, check=False)
        if r.returncode == 0 and (r.stdout or r.stderr):
            time.sleep(2)
            break
    except (FileNotFoundError, subprocess.TimeoutExpired):
        continue

(COMFY_DIR / 'output').mkdir(parents=True, exist_ok=True)
(COMFY_DIR / 'input').mkdir(parents=True, exist_ok=True)

print(f'  Launching: {" ".join(LAUNCH_CMD)}')
log_path = COMFY_DIR / 'comfyui.log'
log_f = open(log_path, 'wb')
proc = subprocess.Popen(
    LAUNCH_CMD,
    cwd=str(COMFY_DIR),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    env=env,
    preexec_fn=os.setsid,
)
print(f'  PID: {proc.pid}, log: {log_path}')

# poll for /system_stats, but check proc.poll() FIRST.
ready = False
deadline = time.time() + 600
while time.time() < deadline:
    if proc.poll() is not None:
        print(f'  ComfyUI exited early with code {proc.returncode}. Last 40 log lines:')
        with open(log_path) as f:
            for ln in f.read().splitlines()[-40:]:
                print('   ', ln)
        raise SystemExit('ComfyUI failed to start.')
    try:
        with urllib.request.urlopen(COMFY_URL + '/system_stats', timeout=2) as r:
            r.read()
        ready = True
        break
    except (urllib.error.URLError, ConnectionResetError, OSError):
        pass
    time.sleep(2)

if not ready:
    proc.terminate()
    raise SystemExit(f'ComfyUI did not respond within 10 minutes. Tail of log:\n'
                     + '\n'.join(open(log_path).read().splitlines()[-20:]))

# Sanity check: verify the key nodes are registered. We rely on:
#   UNETLoader           (ComfyUI core, the transformer goes here)
#   VAELoader             (ComfyUI core, the video + audio VAEs)
#   EmptyLTXVLatentVideo  (ComfyUI v0.31+ core, LTXV latent empty)
#   LTXVConditioning      (ComfyUI v0.31+ core, prompt conditioning)
#   ComfyUI v0.31+ also pulls in comfy_extras/nodes_lt.py
#   automatically (no flag needed); v0.30.0 doesn't have these
#   core LTX nodes and the workflow would fail at validation time.
#   Our pinned install just runs 'git clone' with no version pin so
#   latest master is what we get — usually v0.32.0 as of late 2026.
import requests as _req
# If the user picked a GGUF transformer, city96/ComfyUI-GGUF
# becomes a hard dependency. Probe the GGUF loader too.
_TRANSFORMER_VAR = getattr(__import__("builtins"), "_TRANSFORMER_VAR", None)
_WANTS_GGUF = bool(_TRANSFORMER_VAR and "GGUF" in _TRANSFORMER_VAR)
_loader_set = ['UNETLoader', 'VAELoader', 'EmptyLTXVLatentVideo', 'LTXVConditioning']
_loader_label = "ComfyUI v0.31+ core LTX nodes: ready"
if _WANTS_GGUF:
    _loader_set.append('UnetLoaderGGUF')
    _loader_label += ' + city96 UnetLoaderGGUF'
for _node in _loader_set:
    _r = _req.get(f'{COMFY_URL}/object_info/{_node}', timeout=10)
    if _r.status_code != 200:
        print(f'  WARNING: {_node} not registered. ComfyUI is probably < v0.31.')
        print(f'           Re-run STEP 1 (git pull inside {COMFY_DIR} to fetch a newer main).')
        break
else:
    print('  ComfyUI v0.31+ core LTX nodes: ready (UNETLoader, VAELoader, EmptyLTXVLatentVideo, LTXVConditioning)')

import builtins as _builtins
_builtins.AEI_COMFY_PROC = proc
_builtins.AEI_COMFY_URL = COMFY_URL
_builtins.AEI_COMFY_LOG = log_path
_builtins.AEI_COMFY_DIR = COMFY_DIR
print(f'  Stored AEI_COMFY_PROC / AEI_COMFY_URL in builtins.')
print(f'  ComfyUI ready on {COMFY_URL}. Try the GUI at that URL.')



In [ ]:
#@title STEP 4 — (Optional) Gradio UI for LTX-2.5

"""
Optional UI step. Same pattern as MiniMax-H3 notebook — exposes
prompt + canvas + steps + an audio override via gradio.
"""
# If you don't want the UI, skip this cell and use STEP 6 / STEP 7
# to submit workflows via REST directly (they call _build_workflow()
# the same way STEP 4 does).
print('STEP 4 is optional for LTX-2.5; most users will use STEP 6/7 instead.')
print('See _build_workflow() in STEP 6 for the workflow template.')



In [ ]:
#@title STEP 5 — Keep-alive + session summary (ComfyUI status)

import time, json, urllib.request, builtins

URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188')

import IPython.display
display(IPython.display.Javascript("""
function KeepAlive() { console.log('Colab session kept alive at ' + new Date().toISOString()); }
setInterval(KeepAlive, 60000);
"""))

print('=' * 72)
print('LTX-2.5 / ComfyUI session summary')
print('=' * 72)

try:
    with urllib.request.urlopen(URL + '/system_stats', timeout=5) as r:
        stats = json.loads(r.read())
    devs = stats.get('devices', [])
    for d in devs:
        free  = d.get('vram_free', 0) / 1024**3
        total = d.get('vram_total', 0) / 1024**3
        print(f'  GPU {d.get("name"):<24} {free:5.1f} GB free / {total:5.1f} GB total')
    print(f'  ComfyUI version  : {stats.get("system", {}).get("comfyui_version", "?")}')
    print(f'  Python version   : {stats.get("system", {}).get("python_version", "?")}')
    print(f'  Embedded at      : {URL}')
    print(f'  Log file         : {getattr(builtins, "AEI_COMFY_LOG", "?")}')
    print(f'  Output dir       : {getattr(builtins, "AEI_COMFY_DIR", "?")}/output')
    print()
    print('  Three ways to use:')
    print('    - STEP 6 (quick test): single video via form params, polls /history')
    print('    - STEP 7 (batch): JSON scene list for production runs')
    print('  STEP 8 tails the ComfyUI log if a run stalls.')
except Exception as e:
    print(f'  [WARN] ComfyUI not reachable: {e}')



In [ ]:
#@title STEP 6 — Quick test (single video generation)

"""
Build a single LTX-2.5 t2v workflow and queue it via POST /prompt.
Uses the same _build_workflow pattern as STEP 4 (which is currently
a stub for LTX-2.5; STEP 6 has the full chain). Mirrors the MiniMax-H3
notebook's STEP 6 but with the LTX nodes.
"""

import os, sys, time, json, uuid, shutil, urllib.request, urllib.parse, requests
from pathlib import Path
import builtins
from IPython.display import display, FileLink

URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188')
COMFY_DIR = Path(getattr(builtins, 'AEI_COMFY_DIR',
                          Path('/content/drive/MyDrive/AEI_ComfyUI')))
OUT_DIR = COMFY_DIR / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Pull the transformer choice from STEP 2 globals. The build-
# script-style globals are imported by reference via builtins,
# populated when STEP 2 ran. If STEP 2 was skipped, fall back
# to the int8-convrot Lightricks defaults so this cell still
# works in isolation (smoke-test still loads the same files).
_TRANSFORMER = getattr(builtins, "_TRANSFORMER_VAR",
                                  'Lightricks/LTX-2.5 (distilled int8-convrot)')
_QUANT = getattr(builtins, "_QUANT_VAR", "Q4_K_M")
# GGUF loader detection: TRANSFORMER option that contains
# "GGUF" means we use city96/ComfyUI-GGUF unet path.
_IS_GGUF = "GGUF" in _TRANSFORMER and "ChrisColeTech" not in _TRANSFORMER
_IS_CCTECH = "ChrisColeTech" in _TRANSFORMER
# Map TRANSFORMER -> loader class + filename. Mirrors STEP 2
# _GGUF_FILES tables. Kept small here; if STEP 2 ran, it set
# the precise filename already.
_GGUF_FILES_STEP6 = {
    'realrebelai/LTX-2.5_GGUFs (distilled GGUF)':
        f'LTX-2.5-Distilled-{_QUANT}.gguf',
    'Abiray/LTX-2.5-Distilled-GGUF (distilled GGUF)':
        f'LTX-2.5-Distilled-{_QUANT}.gguf' if _QUANT != 'Q3_K_S'
        else 'LTX-2.5-Distilled-Q3_K_S.gguf',  # Abiray-only
    'ChrisColeTech/LTX-2.5-turbo-GGUF (uses Lightricks distilled GGUF)':
        f'LTX-2.5-Distilled-{_QUANT}.gguf',
}
_SAFETENSORS_FILES = {
    'Lightricks/LTX-2.5 (distilled int8-convrot)':
        'ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors',
    'Lightricks/LTX-2.5 (distilled nvfp4)':
        'ltx-2.5-22b-distilled-transformer-nvfp4.safetensors',
    'guillaume127/LTX-2.5-FP8':
        'ltx-2.5-22b-distilled-transformer-fp8_e4m3fn.safetensors',
}
if _IS_GGUF or _IS_CCTECH:
    _LOADER_CLASS = 'UnetLoaderGGUF' if _IS_GGUF else 'CCTechUnetLoader'
    _UNET_FILENAME = _GGUF_FILES_STEP6[_TRANSFORMER]
else:
    _LOADER_CLASS = 'UNETLoader'
    _UNET_FILENAME = _SAFETENSORS_FILES[_TRANSFORMER]
print('  Loader       : {_LOADER_CLASS} (on {_TRANSFORMER})')
print('  Transformer  : {_UNET_FILENAME}')

print('=' * 72)
print('LTX-2.5 / ComfyUI — single-video quick test')
print('=' * 72)

PROMPT = 'A red fox trotting through a snowy pine forest at dawn, snow crunching underfoot, shallow depth of field, cinematic 35mm film grain, soft ambient sound.'  #@param {type:"string"}
IMG_PATH = ''  #@param {type:"string"}
SEED = 42  #@param {type:"integer"}
LENGTH = 97  #@param {type:"slider", min:9, max:201, step:8}
STEPS = 8  #@param {type:"slider", min:1, max:20, step:1}
CFG = 1.0  #@param {type:"slider", min:1.0, max:7.0, step:0.1}
RESOLUTION = '832 x 480'  #@param ["768 x 432", "832 x 480", "960 x 544", "1152 x 640", "1920 x 1080"]

if SEED <= 0:
    import random
    SEED = random.randint(1, 2**31 - 1)
    print(f'  Random seed: {SEED}')

# Resolutions for LTX-2.5 — must be divisible by 32 and >= 256. We use
# the trained-default 768x432 / 832x480 for short clips to keep VRAM
# low; 1920x1080 is the model's training-native resolution but needs
# ~32 GB VRAM.
_CANVASES = {
    '768 x 432':   (768, 432),
    '832 x 480':   (832, 480),
    '960 x 544':   (960, 544),
    '1152 x 640':  (1152, 640),
    '1920 x 1080': (1920, 1080),
}
_width, _height = _CANVASES[RESOLUTION]
_LENGTH = LENGTH  # frames at 24 fps
print(f'  Prompt   : {PROMPT[:60]}...')
print(f'  Canvas   : {_width} x {_height} ({_LENGTH} frames)')
print(f'  Steps    : {STEPS}')
print(f'  CFG      : {CFG}')
print(f'  Seed     : {SEED}')
print()

import uuid

def _build_workflow(prompt, width, height, length, steps, seed, cfg):
    """Build an LTX-2.5 t2v workflow.

    Topology mirrors Lightricks' official example_workflows/2.5/
    LTX-2.5_T2V_I2V_Single_Stage_Distilled.json (the path that
    works on L4 24 GB):

      [6]   UNETLoader        (transformer: 20 GB int8-convrot distilled)
      [27]  VAELoader         (video VAE: 1.35 GB Conv VAE)
      [28]  VAELoader         (audio VAE: 0.34 GB bf16)
      [10]  LTXVConditioning  (compresses prompt, no Gemma needed)
      [4]   EmptyLTXVLatentVideo (830x480, 97 frames, batch 1)
      [11]  CFGGuider + KSamplerSelect + ManualSigmas + RandomNoise
      [12]  SamplerCustomAdvanced + LTXVAudioVAEDecode + VAEDecodeTiled
      [40]  CreateVideo       (composes A/V into the output clip)
      [92]  SaveVideo         (writes to disk)

    We use UNETLoader + VAELoader (NOT LowVRAMCheckpointLoader)
    because the guillaume127/LTX-2.5-FP8 file is structured as a
    diffusion-only safetensors (state_dict without a 'first_stage_model'
    'cond_stage_model' prefix), which UNETLoader reads from
    models/diffusion_models/. LowVRAMCheckpointLoader's
    super().load_checkpoint() would error on a diffusion-only file.
    The trade-off: we lose the explicit dependencies-load mechanism,
    so on 24 GB cards the workflow expects --highvram (which keeps
    the UNET in VRAM and lets the VAE fit alongside on L4's 22 GB
    headroom). On 24 GB-class cards without --highvram, replace
    guillaume127/LTX-2.5-FP8 with the Lightricks int8-convrot full
    checkpoint and add LowVRAMCheckpointLoader with dependencies.
    """
    p = {}
    # Diffusion model: use UNETLoader (matches Lightricks example).
    # weight_dtype 'default' keeps ComfyUI's autoload logic for fp8/int8.
    p["6"] = {"class_type": "UNETLoader", "inputs": {
        "unet_name": "ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors",
        "weight_dtype": "default",
    }}
    # Two separate VAELoaders — video VAE for the pixel frames, audio
    # VAE for the audio stream. Both loaded from models/vae/.
    p["27"] = {"class_type": "VAELoader", "inputs": {
        "vae_name": "ltx-2.5-video-vae-conv-bf16.safetensors",
    }}
    p["28"] = {"class_type": "VAELoader", "inputs": {
        "vae_name": "ltx-2.5-audio-vae-bf16.safetensors",
    }}
    # Prompt conditioning via LTXVConditioning (compresses prompt and
    # builds the conditioning tensor; does NOT require Gemma 3 14 GB
    # to be loaded — uses ComfyUI's bundled CLIP and the LTXV text
    # encoder intrinsic). For LLM-enhanced prompts, swap this for
    # LTXVGemmaEnhancePrompt + LTXVGemmaCLIPModelLoader (requires the
    # 14 GB text encoder + a way to fit it on 24 GB). GemmaAPITextEncode
    # is the third option (uses Lightricks' hosted Gemma API).
    p["10"] = {"class_type": "LTXVConditioning", "inputs": {
        "prompt": prompt,
    }}
    # Empty latent video — ComfyUI core's LTXV-empty-latent-video node.
    p["4"] = {"class_type": "EmptyLTXVLatentVideo", "inputs": {
        "width": width,
        "height": height,
        "length": length,
        "batch_size": 1,
    }}
    # Sampling — manual CFGGuider + sigmas + KSampler for full control.
    # distilled uses a fixed 8-step schedule at CFG=1; higher CFG mushes
    # the output. Sample with euler/normal scheduler.
    p["5"] = {"class_type": "CFGGuider", "inputs": {
        "model": ["6", 0],
        "positive": ["10", 0],
        "negative": ["10", 0],
        "cfg": cfg,
    }}
    p["7"] = {"class_type": "RandomNoise", "inputs": {
        "noise_seed": seed,
    }}
    p["8"] = {"class_type": "KSamplerSelect", "inputs": {
        "sampler_name": "euler",
    }}
    p["9"] = {"class_type": "ManualSigmas", "inputs": {
        "steps": steps,
        "denoise_min": 0.0,
        "denoise_max": 1.0,
        "sigma_min": 1.0,
        "sigma_max": 0.0,
    }}
    p["11"] = {"class_type": "SamplerCustomAdvanced", "inputs": {
        "noise": ["7", 0],
        "guider": ["5", 0],
        "sampler": ["8", 0],
        "sigmas": ["9", 0],
        "latent_image": ["4", 0],
    }}
    # Decode the video latent with the video VAE, audio latent with
    # the audio VAE; CreateVideo fuses both into a single .mp4.
    p["13"] = {"class_type": "LTXVAudioVAEDecode", "inputs": {
        "samples": ["11", 0],
        "audio_vae": ["28", 0],
    }}
    p["14"] = {"class_type": "VAEDecodeTiled", "inputs": {
        "samples": ["11", 0],
        "vae": ["27", 0],
        "tile_size": 256,
        "overlap": 32,
    }}
    p["40"] = {"class_type": "CreateVideo", "inputs": {
        "images": ["14", 0],
        "audio": ["13", 0],
        "fps": 24,
    }}
    return {"prompt": p}


_slug = re.sub(r'[^a-zA-Z0-9_\-]+', '-', PROMPT).strip('-')[:40]
import re as _re
_slug = _re.sub(r'[^a-zA-Z0-9_\-]+', '-', PROMPT).strip('-')[:40]
ts = int(time.time())
_prefix = (f'video/LTX_quicktest_{_slug}_{_width}x{_height}_f{_LENGTH}'
           f'_s{STEPS}_seed{SEED}_{ts}')

wf = _build_workflow(PROMPT, _width, _height, _LENGTH, STEPS, SEED, CFG)
nodes = wf['prompt']

# SaveVideo needs a filename_prefix field
nodes["92"] = {"class_type": "SaveVideo", "inputs": {
    "video": ["40", 0],
    "filename_prefix": _prefix,
    "format": "auto",
    "codec": "auto",
}}

t0 = time.time()
r = requests.post(URL + '/prompt',
                  json={'prompt': nodes, 'client_id': str(uuid.uuid4())},
                  timeout=60)
if r.status_code != 200:
    raise SystemExit(f'ComfyUI rejected the workflow: {r.status_code} {r.text[:1000]}')
prompt_id = r.json()['prompt_id']
print(f'  Queued: {prompt_id[:8]} ... polling /history')

# Poll for completion.
while True:
    h = requests.get(f'{URL}/history/{prompt_id}', timeout=10).json()
    if prompt_id in h:
        entry = h[prompt_id]
        if entry.get('status', {}).get('completed'):
            elapsed = time.time() - t0
            print(f'  Done in {elapsed:.0f}s ({(elapsed/STEPS):.1f}s/step est).')
            break
        if entry.get('status', {}).get('error'):
            raise gr.Error('ComfyUI failed: ' + json.dumps(entry['status'].get('messages', []), indent=2)[:2000])
        time.sleep(3)

# Resolve output file (ComfyUI strips the directory prefix from the
# filename reported in /history).
import re
paths = []
for node_out in entry.get('outputs', {}).values():
    for out in node_out.get('videos', []):
        fn = out.get('filename')
        if not fn:
            continue
        candidates = [OUT_DIR / fn]
        if '/' not in fn:
            candidates.append(OUT_DIR / 'video' / fn)
        local = None
        for c in candidates:
            if c.exists():
                local = c
                break
        if local is None:
            local = candidates[0]
        paths.append(local)
print(f'\n  Outputs ({len(paths)}):')
for p in paths:
    print(f'    {p}')
video_path = next((str(p) for p in paths if str(p).endswith(('.mp4','.webm','.mov'))), str(paths[0]))
print(f'\n  Open: {video_path}')
display(FileLink(video_path))



In [ ]:
#@title STEP 7 — Batch generation from a JSON scene list

"""
Read a JSON scene list and generate videos for each. Uses the same
_build_workflow() as STEP 6 (duplicated for self-containment per
Colab convention). Same pattern as MiniMax-H3 STEP 7.
"""
import os, sys, time, json, uuid, requests, shutil, urllib.request, urllib.parse, gc
from pathlib import Path
import builtins
from IPython.display import display, FileLink

URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188')
COMFY_DIR = Path(getattr(builtins, 'AEI_COMFY_DIR',
                          Path('/content/drive/MyDrive/AEI_ComfyUI')))
OUT_DIR = COMFY_DIR / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Same loader config as STEP 6 (duplicated for self-containment)
# so this STEP 7 cell can run independently of STEP 6. The
# TRANSFORMER + QUANT choices come from builtins._TRANSFORMER_VAR
# / builtins._QUANT_VAR (set by STEP 2). If STEP 2 was skipped,
# defaults to Lightricks int8-convrot (safetensors UNETLoader).
_TRANSFORMER = getattr(builtins, "_TRANSFORMER_VAR",
                                  'Lightricks/LTX-2.5 (distilled int8-convrot)')
_QUANT = getattr(builtins, "_QUANT_VAR", "Q4_K_M")
_IS_GGUF = "GGUF" in _TRANSFORMER and "ChrisColeTech" not in _TRANSFORMER
_IS_CCTECH = "ChrisColeTech" in _TRANSFORMER
_SAFETENSORS_FILES = {
    'Lightricks/LTX-2.5 (distilled int8-convrot)':
        'ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors',
    'Lightricks/LTX-2.5 (distilled nvfp4)':
        'ltx-2.5-22b-distilled-transformer-nvfp4.safetensors',
    'guillaume127/LTX-2.5-FP8':
        'ltx-2.5-22b-distilled-transformer-fp8_e4m3fn.safetensors',
}
if _IS_CCTECH:
    _LOADER_CLASS = 'CCTechUnetLoader'
    _UNET_FILENAME = f'LTX-2.5-Distilled-{_QUANT}.gguf'
elif _IS_GGUF:
    _LOADER_CLASS = 'UnetLoaderGGUF'
    _QUANT_FILE = 'Q3_K_S' if _QUANT == 'Q3_K_S' else _QUANT
    _UNET_FILENAME = f'LTX-2.5-Distilled-{_QUANT_FILE}.gguf'
else:
    _LOADER_CLASS = 'UNETLoader'
    _UNET_FILENAME = _SAFETENSORS_FILES[_TRANSFORMER]

print('=' * 72)
print('LTX-2.5 / ComfyUI — Batch generation')
print('=' * 72)

BATCH_JSON_PATH = '/content/drive/MyDrive/AEI_3D_Cache/LTX-Video-2.5/batch_scenes.json'  #@param {type:"string"}
SKIP_EXISTING = True  #@param {type:"boolean"}

if not os.path.exists(BATCH_JSON_PATH):
    os.makedirs(os.path.dirname(BATCH_JSON_PATH), exist_ok=True)
    with open(BATCH_JSON_PATH, 'w') as f:
        json.dump([
            {'prompt': 'A red fox trotting through a snowy pine forest at dawn, snow crunching, cinematic.'},
            {'prompt': 'A surfer riding a massive wave at sunset, ocean spray, golden light, slow motion.'},
            {'prompt': 'A hummingbird hovering near a hibiscus flower, soft bokeh, warm morning light.'},
        ], f, indent=2)
    print(f'  Wrote sample batch to {BATCH_JSON_PATH}')

with open(BATCH_JSON_PATH) as f:
    scenes = json.load(f)
print(f'  {len(scenes)} scenes from {BATCH_JSON_PATH}')

def _build_workflow(prompt, width, height, length, steps, seed, cfg):
    """Same as STEP 6 — duplicated for self-containment. Keep in sync.

    Uses UNETLoader + VAELoader (NOT LowVRAMCheckpointLoader) because
    the Lightricks int8-convrot file is a diffusion-only state dict in
    models/diffusion_models/, not a checkpoint bundle. See STEP 6 docstring
    for the full rationale.
    """
    p = {}
    p["6"] = {"class_type": "UNETLoader", "inputs": {
        "unet_name": "ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors",
        "weight_dtype": "default",
    }}
    p["27"] = {"class_type": "VAELoader", "inputs": {
        "vae_name": "ltx-2.5-video-vae-conv-bf16.safetensors",
    }}
    p["28"] = {"class_type": "VAELoader", "inputs": {
        "vae_name": "ltx-2.5-audio-vae-bf16.safetensors",
    }}
    p["10"] = {"class_type": "LTXVConditioning", "inputs": {"prompt": prompt}}
    p["4"] = {"class_type": "EmptyLTXVLatentVideo", "inputs": {
        "width": width, "height": height, "length": length, "batch_size": 1,
    }}
    p["5"] = {"class_type": "CFGGuider", "inputs": {
        "model": ["6", 0], "positive": ["10", 0], "negative": ["10", 0], "cfg": cfg,
    }}
    p["7"] = {"class_type": "RandomNoise", "inputs": {"noise_seed": seed}}
    p["8"] = {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "euler"}}
    p["9"] = {"class_type": "ManualSigmas", "inputs": {
        "steps": steps, "denoise_min": 0.0, "denoise_max": 1.0,
        "sigma_min": 1.0, "sigma_max": 0.0,
    }}
    p["11"] = {"class_type": "SamplerCustomAdvanced", "inputs": {
        "noise": ["7", 0], "guider": ["5", 0], "sampler": ["8", 0],
        "sigmas": ["9", 0], "latent_image": ["4", 0],
    }}
    p["13"] = {"class_type": "LTXVAudioVAEDecode", "inputs": {
        "samples": ["11", 0], "audio_vae": ["28", 0],
    }}
    p["14"] = {"class_type": "VAEDecodeTiled", "inputs": {
        "samples": ["11", 0], "vae": ["27", 0], "tile_size": 256, "overlap": 32,
    }}
    p["40"] = {"class_type": "CreateVideo", "inputs": {
        "images": ["14", 0], "audio": ["13", 0], "fps": 24,
    }}
    p["92"] = {"class_type": "SaveVideo", "inputs": {
        "video": ["40", 0], "filename_prefix": "", "format": "auto", "codec": "auto",
    }}
    return {"prompt": p}

results = []
for i, sc in enumerate(scenes):
    prompt = sc.get('prompt', 'A red fox trotting through a snowy forest at dawn.')
    width = sc.get('width', 832)
    height = sc.get('height', 480)
    length = sc.get('length', 97)
    steps = sc.get('steps', 8)
    cfg = sc.get('cfg', 1.0)
    seed = sc.get('seed', 0)
    if seed <= 0:
        import random
        seed = random.randint(1, 2**31 - 1)
    img = sc.get('image', '').strip() or None

    scene_slug = _re.sub(r'[^a-zA-Z0-9_\-]+', '-', prompt)[:30]
    scene_ts = int(time.time())
    scene_prefix = (f'video/LTX_batch_{i:03d}_{scene_slug}_{width}x{height}_f{length}'
                    f'_s{steps}_seed{seed}_{scene_ts}')

    if SKIP_EXISTING:
        existing = list(OUT_DIR.glob(f'LTX_batch_{i:03d}_*'))
        if existing:
            print(f'  [{i+1}/{len(scenes)}] SKIP (already exists: {existing[0].name})')
            results.append(str(existing[0]))
            continue

    print(f'\n  [{i+1}/{len(scenes)}] {width}x{height} {length}f {steps}steps seed={seed}')
    print(f'    {prompt[:80]}')

    nodes = {}
    wf = _build_workflow(prompt, width, height, length, steps, seed, cfg)
    nodes = wf['prompt']
    nodes["92"]["inputs"]["filename_prefix"] = scene_prefix

    r = requests.post(URL + '/prompt',
                      json={'prompt': nodes, 'client_id': str(uuid.uuid4())},
                      timeout=60)
    if r.status_code != 200:
        print(f'    FAIL: {r.status_code} {r.text[:300]}')
        results.append(None)
        continue
    pid = r.json()['prompt_id']
    print(f'    Queued as {pid[:8]} — polling /history for completion ...')

    while True:
        h = requests.get(f'{URL}/history/{pid}', timeout=10).json()
        if pid in h:
            entry = h[pid]
            if entry.get('status', {}).get('completed'):
                elapsed = time.time() - scene_ts
                print(f'    Done in {elapsed:.0f}s.')
                break
            if entry.get('status', {}).get('error'):
                print('    FAIL')
                break
        time.sleep(3)

    for node_out in entry.get('outputs', {}).values():
        for out in node_out.get('videos', []):
            fn = out.get('filename')
            if not fn:
                continue
            candidates = [OUT_DIR / fn, OUT_DIR / 'video' / fn]
            local = next((c for c in candidates if c.exists()), candidates[0])
            results.append(str(local))
            print(f'    {local}')
            break
        else:
            continue
        break

print()
print(f'Generated {sum(1 for r in results if r)}/{len(results)} clips.')



In [ ]:
#@title STEP 8 — Tail ComfyUI log (debugging aid)

import time
from pathlib import Path

LOG = Path(getattr(__import__('builtins'), 'AEI_COMFY_LOG',
                     '/content/drive/MyDrive/AEI_ComfyUI/comfyui.log'))
TAIL_LINES = 80  #@param {type:"slider", min:10, max:500, step:10}

if LOG.exists():
    lines = LOG.read_text(errors='replace').splitlines()
    print(f'  Last {min(TAIL_LINES, len(lines))} of {len(lines)} log lines from {LOG}:')
    print()
    for line in lines[-TAIL_LINES:]:
        print(f'    {line}')
else:
    print(f'  No log file at {LOG}')



In [ ]:
#@title STEP 8.5 — Smoke-test the workflow wiring

"""
Run a minimal t2v workflow at LENGTH=9, STEPS=2 with a tiny canvas.
Verifies the UNETLoader + VAELoader + LTXVConditioning chain wires
up correctly. Should complete in ~30-60 s on L4 (dominated by
model load time, not the denoise itself).
"""
import json, time, uuid, requests, builtins
from pathlib import Path
URL = getattr(builtins, 'AEI_COMFY_URL', 'http://127.0.0.1:8188')
COMFY_DIR = Path(getattr(builtins, 'AEI_COMFY_DIR',
                          Path('/content/drive/MyDrive/AEI_ComfyUI')))
OUT_DIR = COMFY_DIR / 'output'

w, h, length, steps, cfg = 256, 256, 9, 2, 1.0

# Same loader detection as STEP 6 (duplicated for self-containment).
_TRANSFORMER = getattr(builtins, "_TRANSFORMER_VAR",
                                  'Lightricks/LTX-2.5 (distilled int8-convrot)')
_QUANT = getattr(builtins, "_QUANT_VAR", "Q4_K_M")
_IS_GGUF = "GGUF" in _TRANSFORMER and "ChrisColeTech" not in _TRANSFORMER
_IS_CCTECH = "ChrisColeTech" in _TRANSFORMER
_SAFETENSORS_FILES = {
    'Lightricks/LTX-2.5 (distilled int8-convrot)':
        'ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors',
    'Lightricks/LTX-2.5 (distilled nvfp4)':
        'ltx-2.5-22b-distilled-transformer-nvfp4.safetensors',
    'guillaume127/LTX-2.5-FP8':
        'ltx-2.5-22b-distilled-transformer-fp8_e4m3fn.safetensors',
}
if _IS_CCTECH:
    _LOADER_CLASS = 'CCTechUnetLoader'
    _UNET_FILENAME = f'LTX-2.5-Distilled-{_QUANT}.gguf'
elif _IS_GGUF:
    _LOADER_CLASS = 'UnetLoaderGGUF'
    _QUANT_FILE = 'Q3_K_S' if _QUANT == 'Q3_K_S' else _QUANT
    _UNET_FILENAME = f'LTX-2.5-Distilled-{_QUANT_FILE}.gguf'
else:
    _LOADER_CLASS = 'UNETLoader'
    _UNET_FILENAME = _SAFETENSORS_FILES[_TRANSFORMER]
seed = 12345

# Minimal workflow (matches Lightricks example_workflows/2.5/).
nodes = {
    "6": {"class_type": _LOADER_CLASS, "inputs": {
        "unet_name": _UNET_FILENAME,
        "weight_dtype": "default",
    }},
    "27": {"class_type": "VAELoader", "inputs": {
        "vae_name": "ltx-2.5-video-vae-conv-bf16.safetensors",
    }},
    "28": {"class_type": "VAELoader", "inputs": {
        "vae_name": "ltx-2.5-audio-vae-bf16.safetensors",
    }},
    "10": {"class_type": "LTXVConditioning", "inputs": {
        "prompt": "A fluffy white cloud drifting across a blue sky.",
    }},
    "4": {"class_type": "EmptyLTXVLatentVideo", "inputs": {
        "width": w, "height": h, "length": length, "batch_size": 1,
    }},
    "5": {"class_type": "CFGGuider", "inputs": {
        "model": ["6", 0], "positive": ["10", 0], "negative": ["10", 0], "cfg": cfg,
    }},
    "7": {"class_type": "RandomNoise", "inputs": {"noise_seed": seed}},
    "8": {"class_type": "KSamplerSelect", "inputs": {"sampler_name": "euler"}},
    "9": {"class_type": "ManualSigmas", "inputs": {
        "steps": steps, "denoise_min": 0.0, "denoise_max": 1.0,
        "sigma_min": 1.0, "sigma_max": 0.0,
    }},
    "11": {"class_type": "SamplerCustomAdvanced", "inputs": {
        "noise": ["7", 0], "guider": ["5", 0], "sampler": ["8", 0],
        "sigmas": ["9", 0], "latent_image": ["4", 0],
    }},
    "13": {"class_type": "LTXVAudioVAEDecode", "inputs": {
        "samples": ["11", 0], "audio_vae": ["28", 0],
    }},
    "14": {"class_type": "VAEDecodeTiled", "inputs": {
        "samples": ["11", 0], "vae": ["27", 0], "tile_size": 256, "overlap": 32,
    }},
    "40": {"class_type": "CreateVideo", "inputs": {
        "images": ["14", 0], "audio": ["13", 0], "fps": 24,
    }},
    "92": {"class_type": "SaveVideo", "inputs": {
        "video": ["40", 0],
        "filename_prefix": f"smoketest/LTX_{w}x{h}_s{steps}_{int(time.time())}",
        "format": "auto", "codec": "auto",
    }},
}

print(f'  POST /prompt: {w}x{h}, length={length}, {steps} steps ...')
r = requests.post(URL + '/prompt',
                  json={'prompt': nodes, 'client_id': str(uuid.uuid4())},
                  timeout=60)
if r.status_code != 200:
    raise SystemExit(f'Workflow rejected: {r.status_code}\n{r.text[:1000]}')
pid = r.json()['prompt_id']
print(f'  Queued as {pid[:8]}; polling /history for completion ...')
t0 = time.time()
while True:
    h = requests.get(f'{URL}/history/{pid}', timeout=10).json()
    if pid in h:
        entry = h[pid]
        if entry.get('status', {}).get('completed'):
            print(f'  Done in {time.time() - t0:.0f}s.')
            for node_out in entry.get('outputs', {}).values():
                for out in node_out.get('videos', []):
                    fn = out.get('filename')
                    if fn:
                        print(f'  Output: {OUT_DIR / fn}')
            break
        if entry.get('status', {}).get('error'):
            print('  FAIL:')
            for msg in entry['status'].get('messages', []):
                print(f'    {msg}')
            break
    time.sleep(3)

